# 0: Установка зависимостей

In [1]:
# Установка всех необходимых библиотек
# -q скрывает подробный вывод pip, делая лог чище
!pip install -q datasets transformers torch accelerate evaluate scikit-learn matplotlib seaborn pandas

# 1: Импорты, Seed и определение устройства

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    pipeline
)
import evaluate
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

# 1. Фиксация seed для воспроизводимости (Python, NumPy, PyTorch)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 2. Определение устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Используемое устройство: {device}")
print(f"[✓] Зафиксированный seed: {SEED}")

[✓] Используемое устройство: cpu
[✓] Зафиксированный seed: 42


# 2: Загрузка данных и sanity-check

In [3]:
# Загрузка рекомендованного датасета `emotion` из Hugging Face Datasets.
# Он уже содержит официальные сплиты train/validation/test.
raw_dataset = load_dataset("emotion")

train_ds = raw_dataset["train"]
val_ds = raw_dataset["validation"]
test_ds = raw_dataset["test"]

# Получение имён классов из метаданных датасета
label_names = train_ds.features["label"].names

print("="*50)
print("[ДАННЫЕ] Sanity-check")
print(f"Размеры сплитов: train={len(train_ds)}, validation={len(val_ds)}, test={len(test_ds)}")
print(f"Классы ({len(label_names)} шт.): {label_names}")
print("\n[ПРИМЕРЫ] 5 случайных записей из train-выборки:")
for i in range(5):
    label_idx = train_ds[i]["label"]
    text = train_ds[i]["text"]
    print(f"  [{label_names[label_idx].upper():8s}] {text}")
print("="*50)

[ДАННЫЕ] Sanity-check
Размеры сплитов: train=16000, validation=2000, test=2000
Классы (6 шт.): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

[ПРИМЕРЫ] 5 случайных записей из train-выборки:
  [SADNESS ] i didnt feel humiliated
  [SADNESS ] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
  [ANGER   ] im grabbing a minute to post i feel greedy wrong
  [LOVE    ] i am ever feeling nostalgic about the fireplace i will know that it is still on the property
  [ANGER   ] i am feeling grouchy


# 3: Демонстрация токенизации

In [4]:
# Выбираем стандартный BERT-токенизатор. Он же будет использоваться для обучения.
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Тестовые фразы разной длины для демонстрации работы truncation и padding
sample_texts = [
    "i feel a little stunned but can't imagine what the folks are experiencing today",
    "im feeling positive",
    "this is a very long sentence designed specifically to test truncation because we set a small max length limit and want to see how the tokenizer cuts off the tail end of the sequence"
]

print(f"[ТОКЕНИЗАТОР] {MODEL_NAME}")
print("-" * 80)

for txt in sample_texts:
    # Кодирование с явным указанием padding и truncation
    encoded = tokenizer(
        txt, 
        padding="max_length", 
        max_length=16,       # Намеренно маленькое значение для наглядности
        truncation=True, 
        return_tensors="pt"
    )
    
    tokens = tokenizer.tokenize(txt)
    input_ids = encoded["input_ids"].tolist()[0]
    attention_mask = encoded["attention_mask"].tolist()[0]
    
    print(f"Текст: {txt}")
    print(f"Токены (с обрезкой/паддингом): {tokens[:10]}{'...' if len(tokens)>10 else ''}")
    print(f"Input IDs:           {input_ids}")
    print(f"Attention Mask:      {attention_mask}")
    print(f"Special tokens: [CLS]=101, [SEP]=102, [PAD]=0. Внимание (1) ставится только на реальные токены.")
    print("-" * 80)

[ТОКЕНИЗАТОР] bert-base-uncased
--------------------------------------------------------------------------------
Текст: i feel a little stunned but can't imagine what the folks are experiencing today
Токены (с обрезкой/паддингом): ['i', 'feel', 'a', 'little', 'stunned', 'but', 'can', "'", 't', 'imagine']...
Input IDs:           [101, 1045, 2514, 1037, 2210, 9860, 2021, 2064, 1005, 1056, 5674, 2054, 1996, 12455, 2024, 102]
Attention Mask:      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Special tokens: [CLS]=101, [SEP]=102, [PAD]=0. Внимание (1) ставится только на реальные токены.
--------------------------------------------------------------------------------
Текст: im feeling positive
Токены (с обрезкой/паддингом): ['im', 'feeling', 'positive']
Input IDs:           [101, 10047, 3110, 3893, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask:      [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens: [CLS]=101, [SEP]=102, [PAD]=0. Внимание (1) ставится только на реальн

# 4: Инференс

In [5]:
# Загружаем готовую модель, дообученную на схожей задаче (эммоциональная классификация).
# RoBERTa является BERT-подобной архитектурой, что удовлетворяет условию задания.
PRETRAINED_MODEL = "j-hartmann/emotion-english-distilroberta-base"
classifier = pipeline("text-classification", model=PRETRAINED_MODEL, device=0 if device.type == "cuda" else -1)

test_inference_texts = [
    "I am so incredibly happy today!",
    "This makes me really angry.",
    "I feel a bit scared about tomorrow.",
    "What a wonderful surprise!",
    "I'm feeling quite sad right now."
]

print(f"[ИНФЕРЕНС] Модель: {PRETRAINED_MODEL}")
for txt in test_inference_texts:
    # pipeline возвращает список словарей с меткой и уверенностью
    res = classifier(txt)[0]
    print(f"'{txt}' -> {res['label']} (conf: {res['score']:.3f})")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ИНФЕРЕНС] Модель: j-hartmann/emotion-english-distilroberta-base
'I am so incredibly happy today!' -> joy (conf: 0.975)
'This makes me really angry.' -> anger (conf: 0.972)
'I feel a bit scared about tomorrow.' -> fear (conf: 0.994)
'What a wonderful surprise!' -> joy (conf: 0.663)
'I'm feeling quite sad right now.' -> sadness (conf: 0.989)


# 5: Подготовка датасета для обучения

In [6]:
# Функция токенизации, применяемая ко всему датасету через map
def tokenize_function(examples):
    # max_length=64 покрывает >95% текстов в данном датасете
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

print("[ПОДГОТОВКА] Токенизация датасета")
tokenized_datasets = DatasetDict({
    "train": train_ds.map(tokenize_function, batched=True, remove_columns=["text"]),
    "validation": val_ds.map(tokenize_function, batched=True, remove_columns=["text"]),
    "test": test_ds.map(tokenize_function, batched=True, remove_columns=["text"])
})
print("[✓] Токенизация завершена.")

[ПОДГОТОВКА] Токенизация датасета
[✓] Токенизация завершена.


# 6: Настройка модели, метрик и Trainer

In [7]:
# Инициализация метрик для вычисления во время обучения и валидации
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

# Инициализация модели для Sequence Classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(label_names)
)

# Настройки обучения
training_args = TrainingArguments(
    output_dir="./hw13_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to="none"
)

# Инициализация Trainer (аргумент tokenizer удалён, т.к. данные уже оттокенизированы)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 7: Обучение (Fine-tuning)

In [8]:
print("[ОБУЧЕНИЕ] Запуск fine-tuning...")
train_result = trainer.train()
print("[✓] Обучение завершено.")
print(f"Лучший чекпоинт (по validation f1_macro) загружен автоматически.")
print(f"Метрики на последней эпохе: {train_result.metrics}")

c:\Python\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[ОБУЧЕНИЕ] Запуск fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.604416,0.204780,0.926500,0.902449
2,0.157465,0.153646,0.937000,0.911019
3,0.099894,0.158602,0.936500,0.913701


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Python\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Python\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

[✓] Обучение завершено.
Лучший чекпоинт (по validation f1_macro) загружен автоматически.
Метрики на последней эпохе: {'train_runtime': 9643.7041, 'train_samples_per_second': 4.977, 'train_steps_per_second': 0.156, 'total_flos': 1578723028992000.0, 'train_loss': 0.28725849151611327, 'epoch': 3.0}


# 8: Оценка на тесте, сохранение артефактов, анализ ошибок

In [11]:
print("[ОЦЕНКА] Финальный прогон на test-выборке...")
predictions = trainer.predict(tokenized_datasets["test"])
logits = predictions.predictions
true_labels = predictions.label_ids
pred_labels = np.argmax(logits, axis=-1)

# Считаем метрики явно через sklearn, чтобы полностью избежать конфликтов 
# с внутренними callback-ами transformers и проблем с префиксами словарей метрик.
test_accuracy = accuracy_score(true_labels, pred_labels)
test_f1 = f1_score(true_labels, pred_labels, average="macro")

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1 Macro: {test_f1:.4f}")

# 1. Сохранение матрицы ошибок
os.makedirs("artifacts", exist_ok=True)
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Confusion Matrix (Test Set)", fontsize=14)
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("[✓] Сохранено: artifacts/confusion_matrix.png")

# 2. Сохранение примеров предсказаний в CSV
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
confidences = np.max(probs, axis=1)

df_preds = pd.DataFrame({
    "text": test_ds["text"],
    "true_label": [label_names[l] for l in true_labels],
    "pred_label": [label_names[p] for p in pred_labels],
    "confidence": confidences
})
# Берём репрезентативную выборку из 500 примеров для артефакта
df_sample = df_preds.sample(n=500, random_state=SEED)
df_sample.to_csv("artifacts/sample_predictions.csv", index=False)
print("[✓] Сохранено: artifacts/sample_predictions.csv")

# 3. Краткий анализ ошибок
errors = df_preds[df_preds["true_label"] != df_preds["pred_label"]]
print(f"\nАНАЛИЗ ОШИБОК: Всего неверных предсказаний: {len(errors)}")
print("--- 5 показательных примеров ошибок ---")
for _, row in errors.head(5).iterrows():
    print(f"True: {row['true_label']:8s} | Pred: {row['pred_label']:8s} | Conf: {row['confidence']:.3f}")
    print(f"Text: '{row['text']}'\n")

[ОЦЕНКА] Финальный прогон на test-выборке...


c:\Python\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Accuracy: 0.9245
Test F1 Macro: 0.8758
[✓] Сохранено: artifacts/confusion_matrix.png
[✓] Сохранено: artifacts/sample_predictions.csv

АНАЛИЗ ОШИБОК: Всего неверных предсказаний: 151
--- 5 показательных примеров ошибок ---
True: fear     | Pred: anger    | Conf: 0.637
Text: 'i don t feel particularly agitated'

True: anger    | Pred: sadness  | Conf: 0.843
Text: 'i feel if i completely hated things i d exercise my democratic right speak my mind in what ever ways possible and try to enact a change'

True: anger    | Pred: sadness  | Conf: 0.541
Text: 'i feel a bit stressed even though all the things i have going on are fun'

True: surprise | Pred: fear     | Conf: 0.570
Text: 'i am right handed however i play billiards left handed naturally so me trying to play right handed feels weird'

True: anger    | Pred: fear     | Conf: 0.559
Text: 'when a friend dropped a frog down my neck'

